# Outcomes: 자기 결과물을 스스로 검증하는 에이전트

에이전트는 다 된 것*처럼 보이는* 결과물을 잘 만들어 냅니다. 출처가 달린 리서치 브리프를 요청하면 각주가 붙은 깔끔한 문서를 돌려받습니다. 자세히 들여다보면 대개 개선할 여지가 있습니다. 어떤 주제는 얄팍하게 다뤄지고, 인용문이 원문과 미묘하게 어긋나고, 원 공시 대신 보도 자료를 근거로 삼습니다. 이런 것을 잡으려면 수작업 검토 루프가 필요합니다. 결과물을 읽고, 어긋난 곳을 찾아내고, 다시 프롬프트를 줍니다. 그리고 그 과정에서 하는 말의 대부분은 에이전트가 시작하기 전에 미리 적어 둘 수 있었던 피드백입니다.

Claude Managed Agents의 [Outcomes](https://platform.claude.com/docs/en/managed-agents/define-outcomes)는 검사만 전담하는 두 번째 에이전트를 세션에 붙여 줍니다. "완료"가 어떤 모습이고 어떻게 검증하는지 적어 둔 **루브릭**을 작성하면, 플랫폼이 자체 컨텍스트 윈도를 가진 **채점자**를 띄웁니다. 채점자는 작성자의 추론 과정을 볼 수 없고 어떤 지름길을 택했는지도 모릅니다. 작성자의 턴이 끝날 때마다 루브릭에 비추어 산출물을 다시 읽고, 통과시키거나 기준별 미흡 목록을 돌려줍니다. 작성자가 수정하면 루프가 다시 돌고, 여러분이 정한 상한까지 반복됩니다.

이 가이드에서는 그 루프가 처음부터 끝까지 도는 것을 지켜봅니다. 작성자는 전기차 급속 충전 경제성에 대한 한 쪽짜리 브리프를 출처의 원문 인용과 함께 작성합니다. 채점자는 인용된 URL을 모두 직접 가져와 각 페이지에서 인용 문자열을 찾고, 그 인용이 실제로 주장을 뒷받침하는지 확인하고, 일곱 항목 체크리스트로 커버리지를 채점합니다. 실제 문제(10-K가 요구되는 자리에 보도 자료 첨부물을 인용한 것)를 잡아내는 모습과, 작성자가 올바른 문서를 찾아 나서는 모습을 보게 됩니다.

## 배울 내용

- 채점자가 실제로 행동할 수 있는 루브릭 작성하기
- Outcome과 함께 세션을 시작해 에이전트가 그 목표를 향해 일하게 하기
- 채점-수정 루프를 따라가며 채점자의 피드백 읽기
- Outcomes가 알맞은 도구인 상황 알아보기

## 1. 환경 설정

먼저 SDK를 설치하고 Anthropic 클라이언트를 설정합니다. `define_outcome` 이벤트와 결과 평가 스팬 타입은 Managed Agents 베타의 일부입니다.

In [1]:
%%capture
%pip install -q anthropic python-dotenv

In [2]:
import os
import re
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = anthropic.Anthropic()

## 2. 작성자 만들고 세션 시작하기

다음으로 작성자 에이전트를 만들고 세션을 엽니다. 작성자의 시스템 프롬프트는 출처를 여섯 개 이하로 인용하되 각각 짧은 원문 인용을 함께 달도록 요구합니다. 채점자가 확인할 구체적인 대상이 있어야 하기 때문입니다.

In [3]:
env = client.beta.environments.create(
    name="research-brief",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

writer = client.beta.agents.create(
    name="Research Analyst",
    model=MODEL,
    system="""You are a research analyst. You write one-page business briefs.

Cite every factual claim with an inline footnote [n]. End the brief with a Sources section in this exact format, one entry per line:

[n] "verbatim quote from the page, 25 words or fewer" - Title - URL

Only cite pages you actually fetched and read. The quote must be copied character-for-character from the page. Cite no more than 6 sources total. Pick the strongest; do not pad. Save the brief to /mnt/session/outputs/brief.md.""",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [
                {"name": "web_search"},
                {"name": "web_fetch"},
                {"name": "read"},
                {"name": "write"},
            ],
        }
    ],
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": writer.id, "version": writer.version},
    environment_id=env.id,
    title="Brief: EV fast-charging unit economics",
    betas=BETAS,
)
print(f"Session {session.id}")

Session sesn_011CakRYQjc4NMXdRy4qKZy5


## 3. 채점자가 실제로 행동할 수 있는 루브릭 작성하기

`define_outcome` 이벤트는 세션에 두 가지를 전달합니다. 작성자가 읽는 산출물 **설명(description)**, 그리고 채점자가 읽는 **루브릭(rubric)**입니다. 작성자의 턴이 끝날 때마다 플랫폼은 작성자와 같은 모델·도구를 가진 새 채점자를 띄우고, 루브릭을 준 뒤 산출물을 검사하게 합니다. 채점자는 기준별 판정을 반환하고, 무엇이든 실패하면 그 설명이 곧바로 작성자에게 전달되어 다음 수정에 반영됩니다.

루브릭은 채점자에 작용하는 유일한 지렛대이며, 어떻게 표현하느냐가 채점자가 실제로 무언가를 확인하는지를 좌우합니다. 기본 실패 양상은 모든 것을 통과시키는 채점자입니다. *"브리프가 수요 요금을 다루는지 확인하라"*고만 적힌 루브릭은 채점자가 브리프를 훑고 수요 요금에 대한 문단을 발견한 뒤 그것이 있다고 한 문장 쓰게 만듭니다. 출처를 한 번도 열지 않고 그 모든 것을 할 수 있습니다. 대부분의 초안이 그렇게 통과하고 루프는 아예 돌지 않습니다. *"브리프를 열고, 수요 요금 섹션을 찾아, $/kW 수치나 운영비 대비 % 수치를 제시하는지 확인하라"*고 적힌 루브릭은 채점자가 증거를 만들어 내게 합니다. 10-K인 척하는 보도 자료를 잡아내는 것이 바로 그런 채점자입니다.

어떤 루브릭에서든 해 둘 만한 것 몇 가지:

| 원칙 | 실제 적용 |
|---|---|
| **기준을 확인 가능하게 만들기** | 과제는 "지명된 사업자의 경제성"이라고 말합니다. 루브릭은 이를 못 박습니다. 10-K나 10-Q의 GAAP 순손실이어야 하고, `sec.gov`의 공시를 인용해야 한다고요. 루브릭은 항상 과제보다 구체적이어야 합니다. |
| **`satisfied`를 채점자가 벌어서 얻게 하기** | 무엇이든 통과시키기 전에 구체적인 증거(가져온 페이지, 추적한 산식, `file:line` 참조)를 요구하세요. 지나치게 엄격한 채점자는 루프를 한 번 더 돌게 하는 비용이 듭니다. 지나치게 느슨한 채점자는 나쁜 버전을 그대로 둔 채 루프를 끝냅니다. |
| **단계가 아니라 목표를 기술하기** | 특정 명령을 지시하는 루브릭은 그 명령을 쓸 수 없을 때 조용히 실패하고, 정작 원했던 확인은 이뤄지지 않습니다. 무엇이 증거로 인정되는지를 정의하세요. 채점자는 작성자의 도구 전체를 갖고 있어 스스로 방법을 찾습니다. |
| **작성자의 지름길을 예상하기** | "미러, 재게시물, 검색 스니펫으로 교차 확인하지 말 것." 이 줄이 없으면 죽은 출처가 스크래퍼 페이지로 바뀌고 채점자가 그것을 통과시킵니다. |
| **피드백 형식을 강제하기** | 채점자의 설명이 작성자가 받는 유일한 신호입니다. 한 줄짜리 점수판을 요구하고, 실패 항목마다 무엇이 잘못됐고 무엇을 해야 하는지 한 줄씩 적게 하세요. |
| **무엇을 무시할지 알려 주기** | 무시 목록이 없으면 채점자는 문체 트집, 기존 문제, 범위 확장에 매달립니다. 무엇이 범위 밖인지 명시하고, 지적하기 전에 스스로 점검하게 하세요. |

> **아직 루브릭이 없다면?** 좋다고 인정되는 산출물 예시를 Claude에 주고 무엇이 그것을 좋게 만드는지 분석하게 한 뒤, 그 분석을 기준으로 바꾸세요. 백지에서 기준을 쓰는 것보다 이 중간 지점이 안정적으로 낫습니다.

다음은 우리 브리프를 위한 과제와 루브릭입니다. 나란히 읽으면서 루브릭이 얼마나 더 구체적인지 보세요. 과제에 없던 세부를 더한 지점마다 채점자가 실제로 실행할 수 있는 확인 항목이 됩니다.

In [4]:
TASK = """Write a brief on the unit economics of public DC fast charging in the United States.
The brief should cover:
  1. Capex range
  2. Demand charges
  3. Utilization breakeven
  4. Subsidy programs
  5. Named-operator economics
  6. A contrarian or skeptical source
  7. Hardware vs install cost split
"""


RUBRIC = """
You are reviewing a research brief at /mnt/session/outputs/brief.md against a coverage checklist and verifying its citations. The writer was told the seven topics to cover; this rubric defines what counts as sufficient coverage for each topic, and how to verify citations.

COVERAGE CHECKLIST. Each item has a specific area:
  1. Capex range: a dollar range for installed cost per DC fast-charging stall or station.
  2. Demand charges: quantified impact on opex (a $/kW figure or a % of operating cost).
  3. Utilization breakeven: a breakeven or target utilization threshold (% or kWh/day).
  4. Subsidy programs: NEVI or another public funding program, named.
  5. Named operator: the GAAP net income or net loss from a specific public charging operator's most recent 10-K or 10-Q, and the citation for it must be the SEC filing itself (sec.gov), not a press release, earnings-call recap, or news article.
  6. Contrarian source: at least one cited source whose thesis is that the economics are unfavorable or structurally challenged.
  7. Cost split: a hardware vs soft-cost (install, permitting, grid) breakdown or ratio.

CITATION CHECK. For every [n] entry in the Sources section:
  a. LIVE: Fetch the URL with web_fetch. Mark LIVE only if web_fetch returns the readable page directly. Mark DEAD if 404, parked, login-walled, paywalled, returns a bot-block/403, or renders only via JavaScript. Do NOT corroborate via mirrors, reposts, or search snippets; the cited URL itself must fetch.
  b. VERBATIM: Search the fetched page for the quoted string. Mark QUOTE_MATCH if the exact string appears (treat curly vs straight quotes as equivalent); NOT_FOUND otherwise.
  c. SUPPORTS CLAIM: Mark SUPPORTS_CLAIM if the quoted passage actually backs the claim it's cited on in the brief; UNSUPPORTED if it's tangential, contradicts the claim, or is just a general statement of fact.

OUTPUT FORMAT:

Line 1: Coverage N/7. Citations M/K verified.

Then, for each failed item in the coverage checklist, create a new bullet, name the item and say what specific bar it failed in one sentence max per bullet. For example: "Item 3 Utilization breakeven - MISSING. <what's missing>".

Then, for each failed citation, create a new bullet with the format: "[n] domain - REASON. <what's wrong and what to do>". One sentence max per bullet. For example: "[3] evgo.com - DEAD. The URL returns a 403 error and appears to be behind a bot block. No mirrors or reposts; the cited URL itself must fetch."
"""

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[
        {
            "type": "user.define_outcome",
            "description": TASK,
            "rubric": {"type": "text", "content": RUBRIC},
            "max_iterations": 5,
        },
    ],
)

BetaManagedAgentsSendSessionEvents(data=[BetaManagedAgentsUserDefineOutcomeEvent(id='sevt_011CakRYTKNkXJN7eh4kRs1r', description='Write a brief on the unit economics of public DC fast charging in the United States.\nThe brief should cover:\n  1. Capex range\n  2. Demand charges\n  3. Utilization breakeven\n  4. Subsidy programs\n  5. Named-operator economics\n  6. A contrarian or skeptical source\n  7. Hardware vs install cost split\n', max_iterations=5, outcome_id='outc_011CakRYTKNkJ83CBexcz13e', processed_at=datetime.datetime(2026, 5, 6, 1, 10, 30, 512206, tzinfo=TzInfo(0)), rubric=BetaManagedAgentsTextRubric(content='\nYou are reviewing a research brief at /mnt/session/outputs/brief.md against a coverage checklist and verifying its citations. The writer was told the seven topics to cover; this rubric defines what counts as sufficient coverage for each topic, and how to verify citations.\n\nCOVERAGE CHECKLIST. Each item has a specific area:\n  1. Capex range: a dollar range for insta

위 호출에 대한 몇 가지 참고 사항:

- **설명과 루브릭은 역할이 다릅니다.** 설명은 무엇을 만들지 작성자에게 알려 주고, 루브릭은 어떻게 확인할지 채점자에게 알려 줍니다. 산출물의 위치와 형식에 대해서는 둘이 일치해야 합니다. 서로 모순되면(예를 들어 설명은 인라인 출력을 요구하는데 루브릭은 `/mnt/session/outputs/`의 파일을 채점한다면) 루프는 헤매는 대신 `failed`를 반환합니다.
- **`max_iterations`의 기본값은 3(최대 20)입니다.** 이번 루브릭이 엄격해서 작성자에게 여유가 필요하므로 5로 설정했습니다. 매번 상한에 도달하면서 채점자가 같은 종류의 문제를 계속 찾아낸다면, 작성자가 피드백을 반영하지 못하는 것이고 수렴하지 않는 반복에 비용을 치르고 있는 것입니다.
- **인라인 루브릭 vs. 파일 루브릭.** 노트북에서는 인라인 텍스트로 충분합니다. 프로덕션에서는 Files API로 루브릭을 한 번 업로드하고 `rubric: {"type": "file", "file_id": ...}`을 넘겨, 여러 세션에서 재사용하고 코드처럼 검토할 수 있게 합니다.

**루브릭을 그냥 시스템 프롬프트에 넣으면 안 될까요?** 넣어도 되고, 작성자가 더 잘 겨냥하는 데 도움이 됩니다. 하지만 기준을 아는 작성자는 여전히 자기 결과물을 자기가 채점하는 셈입니다. 스스로 통과했다고 믿으면 통과했다고 말할 것이고, 이미 인용한 URL을 다시 가져오지도, 기억하는 인용문이 페이지의 인용문과 살짝 다르다는 것을 알아차리지도 않을 것입니다. 채점자는 그 확인을 하지 않을 도리가 없습니다. 새 컨텍스트 윈도에 루브릭과 산출물만 갖고 시작하며, 모든 기준에 대해 판정을 내놓기 전까지 플랫폼이 루프를 진행시키지 않습니다. 아무리 잘 쓴 단일 프롬프트로도 그런 분리는 얻을 수 없습니다.

## 4. 검토 루프 지켜보기

이벤트를 스트리밍하며 각 단계를 그때그때 표시해 보겠습니다. 작성자가 초안을 끝내면 배너를 출력하고, 평가가 끝날 때마다 채점자의 피드백을 보여 줍니다.

아래 루프를 위한 표시용 헬퍼 두 개입니다. `banner`는 라벨이 붙은 구분선을 그리고, `render_feedback`은 서버가 각 채점 설명을 감싸는 상용구를 걷어냅니다.

In [5]:
HR = "━" * 46


def banner(label, tag=""):
    print(f"\n{HR}\n{label}  {tag}".rstrip())


def render_feedback(fb: str):
    # Strip the server's per-criterion wrapper and trailer.
    s = re.sub(
        r"^An independent grader found.*?:\n\n- .*?\((?:partially |not )?met\): ",
        "",
        fb,
        count=1,
        flags=re.S,
    )
    s = re.sub(r"\n\nPlease revise your work.*$", "", s, flags=re.S)
    print(s)

In [6]:
TERMINAL = {"satisfied", "max_iterations_reached", "failed", "interrupted"}
t0, res, iters = time.time(), None, 0
n_search, last_len = 0, 0

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            if ev.name in ("web_search", "web_fetch"):
                n_search += 1
            if ev.name == "write" and ev.input["file_path"].endswith("brief.md"):
                last_len = len(ev.input["content"])
        elif ev.type == "span.outcome_evaluation_start":
            banner("writer · " + ("draft" if iters == 0 else f"revision {iters}"))
            print(f"searched/fetched {n_search}× · wrote brief.md ({last_len:,} chars)")
            n_search = 0
        elif ev.type == "span.outcome_evaluation_end":
            res = ev.result
            banner(
                f"grader · pass {iters}",
                "✓ satisfied" if res == "satisfied" else "⟳ needs_revision",
            )
            render_feedback(ev.explanation)
            iters += 1
            if res in TERMINAL:
                break

m, s = divmod(int(time.time() - t0), 60)
print(f"\ndone: {res} after {iters} pass{'es' if iters != 1 else ''} · {m}m {s:02d}s")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
writer · draft
searched/fetched 18× · wrote brief.md (7,607 chars)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
grader · pass 0  ⟳ needs_revision
The brief covers 5 of 7 required topics adequately. Item 2 (Demand charges) fails the quantification bar: the brief describes demand charges qualitatively as the 'single largest operational wildcard' but never states a $/kW figure (McKinsey's footnote of $20/kW is never quoted in the text) or a % of operating cost. Item 5 (Named operator) fails the citation requirement: EVgo's Q1 2024 net loss of $28.2M is cited to evchargingstations.com [6], a third-party news article, not an SEC filing from sec.gov as the rubric explicitly requires. All 6 citations are LIVE, and all 6 quoted strings match the fetched pages and support the claims they are attached to — but [6] is the wrong source type for the Named Operator criterion.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
writer · revision 1
search

### 방금 무슨 일이 있었나

루프는 세 번의 채점을 거쳤습니다.

첫 초안은 일곱 항목 중 다섯을 다뤘습니다. 채점자는 두 가지 누락으로 되돌려 보냈습니다. 수요 요금이 kW당 달러 수치 없이 정성적으로만 서술되었고, 지명 사업자 섹션이 SEC 공시가 아니라 제3자 기사를 인용했습니다. 작성자는 $20/kW 수치를 추가하고 EVgo의 FY2024 순손실을 찾으러 sec.gov로 갔습니다.

두 번째 채점에서는 일곱 중 여섯을 통과했습니다. 작성자가 인용한 sec.gov 문서는 8-K의 첨부물 99.1, 즉 공시된 실적 보도 자료였지 루브릭이 요구한 10-K나 10-Q가 아니었습니다. 채점자는 URL을 읽고 그것이 보도 자료 첨부물임을 식별해, 정확히 그 구분에 근거해 반려했습니다. 세 번째 채점에서 작성자는 EDGAR에서 EVgo의 실제 10-K를 찾아냈고 채점자가 통과시켰습니다. 커버리지 7/7, 인용 6/6이 살아 있고, 인용문이 일치하며, 주장을 뒷받침했습니다.

두 번의 반려 모두 루브릭이 채점자가 확인할 수 있는 선을 그었기에 가능했습니다. "$/kW 수치 또는 운영비 대비 %"와 "보도 자료가 아니라 SEC 공시 그 자체"는 루브릭 작성자가 내린 결정입니다. 과제 자체("수요 요금을 다룰 것", "지명 사업자의 경제성")였다면 둘 다 그냥 통과시켰을 것입니다. 두 번째 반려가 눈여겨볼 만합니다. 작성자는 겉보기에 과제를 충족하는 `sec.gov` URL을 찾아냈는데도, 루브릭이 8-K 보도 자료 첨부물과 10-K를 구분했기 때문에 채점자가 되돌려 보냈습니다. 그 구분이 없었다면 루프는 한 번 일찍 끝나고 더 나쁜 브리프가 남았을 것입니다.

**참고:** 위 기록은 이 쿡북을 처음 실행했을 때의 것입니다. 직접 실행하면 결과가 다를 가능성이 큽니다.

## 5. 최종 브리프 읽기

마지막으로 작성자가 만들어 낸 `brief.md`의 최종 버전을 가져와 무엇을 다루고 무엇을 인용했는지 살펴보겠습니다. 본문 전체를 읽고 싶다면 `content` 자체를 출력하세요.

In [7]:
# Reconstruct the final brief from the event log: full content on `write`,
# then apply each `edit` (old_string -> new_string) in order.
content = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if ev.type != "agent.tool_use" or "brief.md" not in str(ev.input.get("file_path", "")):
        continue
    if ev.name == "write":
        content = ev.input["content"]
    elif ev.name == "edit":
        content = content.replace(ev.input["old_string"], ev.input["new_string"], 1)

# Show the structure and sources rather than the full prose.
for line in content.splitlines():
    if line.startswith(("#", "[")):
        print(line)

# Unit Economics of Public DC Fast Charging in the United States
## 1. Capex Range
## 2. Demand Charges
## 3. Utilization Breakeven
## 4. Subsidy Programs
## 5. Named-Operator Economics
## 6. Contrarian / Skeptical View
## 7. Hardware vs. Installation Cost Split
## Sources
[1] "A 150 to 350kW DCFC charging unit can cost anywhere from $45,000 to over $100,000, and installation costs can range from $40,000 to over $150,000." - Can public EV fast-charging stations be profitable in the United States? - https://www.mckinsey.com/features/mckinsey-center-for-future-mobility/our-insights/can-public-ev-fast-charging-stations-be-profitable-in-the-united-states
[2] "Waiving demand charges for fast chargers will shift costs to all ratepayers, including non-EV drivers." - Fast charging, high costs: Eliminating demand charges won't solve the problem - https://www.utilitydive.com/news/eliminating-demand-charges-wont-solve-EV-station-problems/689395/
[3] "the average total NEVI project cost is $915,42

## 배운 것

- **Outcomes는 좋은 결과가 어떤 모습인지 적어 둘 수 있을 때 알맞습니다.** 세부 사항을 꼼꼼히 따져야 하는 과제나 빠짐없는 커버리지가 필요한 과제가 가장 잘 맞고, 루브릭이 기준을 못 박아 준다면 주관적 품질에도 통합니다.
- **루브릭은 채점자가 증거를 만들어 내게 해야 합니다.** 그러지 않으면 보여 주는 것을 그대로 승인합니다.
- **채점자는 독립적이고 상태가 없습니다.** 자체 컨텍스트 윈도에서 실행되므로 작성자가 설득할 수 없고, 매 반복마다 새 채점자가 산출물 전체를 다시 확인합니다.
- **한 번에 하나의 outcome이지만 이어 붙일 수 있습니다.** 루프가 끝나면 세션은 다시 대화형이 되고, 다음 `user.define_outcome`이 새 루프를 시작합니다.

더 알아보려면 [Outcomes 문서](https://platform.claude.com/docs/en/managed-agents/define-outcomes)를 참고하세요.